# Lab 20 — DPO and Distillation Concepts
**Goal:** Compute a toy DPO-like preference loss and KL divergence for teacher/student distributions.

Free-first: CPU or free Colab where possible; expensive paths are clearly marked optional.



In [ ]:
import math
lp,lr=-0.3,-1.7
rp,rr=-0.5,-1.0
beta=0.1
margin=(lp-rp)-(lr-rr)
loss=-math.log(1/(1+math.exp(-beta*margin)))
print("toy preference loss:",loss)


In [ ]:
teacher=[0.05,0.15,0.70,0.10]
student=[0.10,0.20,0.55,0.15]
kl=sum(t*math.log(t/s) for t,s in zip(teacher,student))
print("KL teacher||student:",kl)


In [ ]:
# Ask: which token receives the largest correction under distillation?
diff=[t-s for t,s in zip(teacher,student)]
print("teacher-student:",diff)


### Answer
DPO uses relative preference data directly in an objective derived from a reward-model formulation. Distillation transfers a teacher's behavior or distribution to a student. The best method follows the supervision signal and the desired endpoint.


## Primary source
https://arxiv.org/abs/2305.18290


## Deepening: preference versus imitation

This section upgrades the notebook from a runnable demo to a measurable experiment. Predict before executing.


In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


In [ ]:
import torch

beta = 0.1
policy_chosen, policy_rejected = -2.0, -3.0
ref_chosen, ref_rejected = -2.2, -2.8
margin = (policy_chosen-policy_rejected) - (ref_chosen-ref_rejected)
loss = -torch.nn.functional.logsigmoid(torch.tensor(beta * margin))
print("toy relative margin:", margin)
print("toy preference loss:", loss.item())

teacher = torch.tensor([0.7, 0.2, 0.1])
student = torch.tensor([0.4, 0.4, 0.2])
kl = torch.sum(teacher * torch.log(teacher/student))
print("toy teacher-student KL:", kl.item())


## Break experiment

Construct a preference dataset where the chosen answer is always longer. Predict the proxy the optimizer may learn.

Then construct pairs that differ in correctness instead of length.

### Report

Explain how preference labels can encode unintended objectives.

Use templates/EXPERIMENT_CARD.md to record the run.


## 7. Research exercise — construct a preference signal from real reasoning data


In [ ]:
from datasets import load_dataset
import random

ds=load_dataset("open-r1/OpenR1-Math-220k",split="train",streaming=True)
rows=[]
for row in ds:
    if row.get("problem") and row.get("solution"):
        rows.append(row)
    if len(rows)>=100: break

random.seed(7)
for r in rows[:5]:
    problem=str(r["problem"])
    solution=str(r["solution"])
    # A deliberately weak negative: truncate the reference reasoning.
    cut=max(1,len(solution)//2)
    chosen=solution
    rejected=solution[:cut]
    print("\nproblem:",problem[:180])
    print("chosen length:",len(chosen),"rejected length:",len(rejected))
    print("pair is suitable only as a teaching example; truncation is not a validated preference label.")


<details><summary><strong>Expected findings / answer</strong></summary>

A preference pair needs a defensible reason why one response is preferred. Truncating a correct solution creates a controlled negative, but it is only a synthetic teaching signal; it does not prove the truncated answer is wrong or less useful. Real preference datasets require annotation criteria or a verifier. Distillation similarly depends on the quality of the teacher signal. The key lesson is **signal validity before optimization method**.
</details>
